# Anime Recommender · Notebook 4 (期末新增): 多模態文字嵌入

**目標**:用 `sentence-transformers` 把每部動漫的 synopsis 轉成 384 維語意向量,
作為「文字模態 (text modality)」存進 artifacts。

## 為什麼這算多模態

- **模態 1 (結構化)**:Genre、Type、Members 數 — 由 Notebook 02 處理
- **模態 2 (文字語意,本 notebook)**:Synopsis 經 sentence-transformer 嵌入 — 384 維向量
- **LLM 角色**:sentence-transformer 本身是 transformer-based 預訓練語言模型;
  後續 Streamlit 端會再呼叫 Groq Llama 3 做推薦解釋

## 重點:不訓練、不需 GPU

sentence-transformers 是預訓練模型,我們只呼叫 `.encode(...)`。
在 Colab CPU 上跑 ~17K 部動漫的 synopsis 約 3–8 分鐘。

## Step 1 — 掛載 Drive + 讀取 Notebook 01 的產物

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import pickle

ARTIFACTS_DIR = '/content/drive/MyDrive/anime-recsys/artifacts'
anime_meta = pd.read_parquet(f'{ARTIFACTS_DIR}/anime_meta.parquet')
with open(f'{ARTIFACTS_DIR}/mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)
anime_id_to_idx = mappings['anime_id_to_idx']
idx_to_anime_id = mappings['idx_to_anime_id']
N_ITEMS = len(anime_id_to_idx)
print(f'#items = {N_ITEMS:,}')
print(f'有 synopsis 的比例: {(anime_meta["synopsis"].str.len() > 50).mean():.2%}')

## Step 2 — 安裝 sentence-transformers

In [ ]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
print('✅ sentence-transformers ready')

## Step 3 — 載入預訓練模型

選用 `all-MiniLM-L6-v2`:
- 384 維輸出 (檔案小,本地端 inference 快)
- 約 80MB,下載 30 秒內
- 英文表現好(我們的 synopsis 主要是英文)
- 在 Colab CPU 上 ~17K 句子大概 3–8 分鐘

In [ ]:
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
model = SentenceTransformer(MODEL_NAME)
print(f'模型: {MODEL_NAME}')
print(f'嵌入維度: {model.get_sentence_embedding_dimension()}')

## Step 4 — 準備文字輸入

對每部動漫,把 name + genre + synopsis 串成一段「語意文本」做嵌入。
即使 synopsis 缺失,name + genre 仍能提供一些訊號。

In [ ]:
# 按 anime_id_to_idx 的順序排列,確保 embedding[i] 對應 idx_to_anime_id[i]
meta_indexed = anime_meta.set_index('anime_id').reindex(list(anime_id_to_idx.keys()))

def build_text(row):
    parts = []
    name = str(row.get('name', '')).strip()
    if name:
        parts.append(f"Title: {name}")
    genre = str(row.get('genre', '')).strip()
    if genre and genre != 'nan':
        parts.append(f"Genres: {genre}")
    atype = str(row.get('type', '')).strip()
    if atype and atype != 'nan':
        parts.append(f"Type: {atype}")
    synopsis = str(row.get('synopsis', '')).strip()
    if synopsis and synopsis != 'nan' and synopsis.lower() != 'no synopsis information has been added to this title':
        parts.append(f"Synopsis: {synopsis}")
    return ' | '.join(parts)

texts = [build_text(row) for _, row in meta_indexed.iterrows()]
# 範例
for i in [0, 1, 2]:
    print(f'--- idx {i} ---')
    print(texts[i][:400])
    print()
print(f'共 {len(texts)} 段文本待嵌入')

## Step 5 — 跑 sentence-transformer 嵌入(主要耗時段)

In [ ]:
import time
t0 = time.time()
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,  # 已 L2 正規化,之後做 cosine 等同點積
)
embeddings = embeddings.astype(np.float32)
elapsed = time.time() - t0
print(f'\n✅ Done in {elapsed/60:.1f} min')
print(f'embeddings shape: {embeddings.shape}, dtype: {embeddings.dtype}')

## Step 6 — 抽查:對指定動漫找文字語意上最相似的 5 部

In [ ]:
def find_similar(name_substr, k=5):
    mask = meta_indexed['name'].str.contains(name_substr, case=False, na=False)
    if not mask.any():
        print(f'找不到包含 "{name_substr}" 的作品')
        return
    idx = np.where(mask.values)[0][0]
    query_name = meta_indexed.iloc[idx]['name']
    print(f'查詢:{query_name} (idx={idx})')
    sims = embeddings @ embeddings[idx]
    top = np.argsort(-sims)[:k+1]
    print('最相似 (按文字語意):')
    for j in top:
        if j == idx:
            continue
        print(f'  [{sims[j]:.3f}] {meta_indexed.iloc[j]["name"]} ({meta_indexed.iloc[j]["genre"]})')

find_similar('Naruto')
print()
find_similar('Attack on Titan')
print()
find_similar('Spirited Away')

## Step 7 — 儲存 artifacts

In [ ]:
np.save(f'{ARTIFACTS_DIR}/text_embeddings.npy', embeddings)
# 額外存一份模型資訊 (給 about 頁面 / 報告引用)
import json
with open(f'{ARTIFACTS_DIR}/text_embeddings_info.json', 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'dim': int(embeddings.shape[1]),
        'n_items': int(embeddings.shape[0]),
        'normalized': True,
    }, f, indent=2)
print(f'✅ Saved text_embeddings.npy ({embeddings.nbytes / 1024 / 1024:.1f} MB)')
!ls -lh {ARTIFACTS_DIR}

## ✅ 完成

產出:
- `text_embeddings.npy` ── (N_ITEMS, 384) 多模態文字嵌入矩陣
- `text_embeddings_info.json` ── 模型資訊

接下來:**把 `MyDrive/anime-recsys/artifacts/` 整個資料夾下載到本地專案的 `artifacts/`**,
然後等我把 `MultimodalRecommender` 和 Groq LLM 解釋功能寫到本地 `src/recommender.py` 和 `app.py`,
就可以跑新的多模態 + LLM Demo 了。